# NO$_2$ concentration with variable values and satellite imagery

Generates paper-ready figures combining a satellite basemap (left) with the TROPOMI tropospheric NO$_2$ column (right) and an annotated key-variable box for each selected power-plant snapshot.

**Inputs**
- `EMISSIONS_CSV` — `updated_tropomi_hourly_emissions_full_variables.csv`
- `PLANTS_CSV` — power-plants CSV with `Latitude`, `Longitude`, `Facility Name`
- (optional) `filtered_df` — DataFrame whose `.index` selects rows; otherwise the first `MAX_PLOTS` rows are used.

**Output** — one PDF per row in `OUTPUT_DIR`.

## 1. Imports and font setup

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import netCDF4 as nc
import contextily as ctx
from pyproj import Geod
from matplotlib import font_manager as fm
from matplotlib.ticker import ScalarFormatter

plt.rcdefaults()
plt.rcParams['figure.facecolor'] = 'white'

def find_font(needles):
    for path in fm.findSystemFonts():
        pl = path.lower()
        if any(n in pl for n in needles) \
                and 'bold' not in pl and 'italic' not in pl and 'oblique' not in pl:
            return path
    return None

# Nimbus Roman (Times-equivalent) is used for everything in the figure including
# the variable annotation box. Reviewer comment: monospace fonts (DejaVu Sans
# Mono, Nimbus Mono PS / Courier, etc.) all render "0" with a dot or slash; the
# serif Nimbus Roman has a plain oval "0".
nimbus_path = find_font(['nimbusroman', 'nimbus_roman'])
prop = fm.FontProperties(fname=nimbus_path, size=12) if nimbus_path else fm.FontProperties(size=12)

print('Font:', nimbus_path)


Font: /usr/share/fonts/urw-base35/NimbusRoman-Regular.otf


## 2. Configuration and data loading

In [ ]:
# Paper revision: 100m localized version (ERA5 100m wind for plume detection,
# LST-corrected NOx join — see 2026-04-28 fix to augment_final_table.py that
# replaced the DST-aware IANA conversion with a fixed winter-offset LST
# conversion to match EPA CAMPD's reporting convention).
# wind_speed in this CSV is already sqrt(wind_u^2 + wind_v^2) of the 100m wind
# components, and file_path is already absolute.
EMISSIONS_CSV = '/net/fs06/d3/rzhuang/TROPOMI/pipeline_100m_run/Run_100m_20260414/updated_tropomi_hourly_emissions_full_variables_augmented_localtz.csv'
PLANTS_CSV    = '/net/fs06/d3/rzhuang/TROPOMI/data/us/power_plants_with_combined_nearby_stats_parallel_debug.csv'
OUTPUT_DIR    = '/net/fs06/d3/rzhuang/TROPOMI/results/paper_figures'

# Older emissions CSVs stored swath paths relative to the original notebook's CWD
# (e.g. '../data/TROPOMI_2019-2024/...'); the augmented_localtz CSVs already store
# absolute paths so this resolver is a no-op for them, but we keep it for
# backward compatibility.
DATA_ROOT = '/net/fs06/d3/rzhuang/TROPOMI/data/us'

MAX_PLOTS = 80
BOX_DEG   = 1.0
PAD       = 0
LBS_TO_KG = 0.453592
ZOOM_KM   = 50  # Satellite image zoom (km half-width)

os.makedirs(OUTPUT_DIR, exist_ok=True)

df     = pd.read_csv(EMISSIONS_CSV)
plants = pd.read_csv(PLANTS_CSV)
print(f'Loaded {len(df):,} emission rows and {len(plants):,} plants')

def resolve_swath_path(file_path):
    """Map a CSV-stored relative swath path to an absolute one under DATA_ROOT."""
    if isinstance(file_path, bytes):
        file_path = file_path.decode()
    if os.path.isabs(file_path) and os.path.exists(file_path):
        return file_path
    p = file_path
    while p.startswith('../'):
        p = p[3:]
    if p.startswith('data/'):
        p = p[len('data/'):]
    return os.path.join(DATA_ROOT, p)


## 3. Pick which rows to plot

If a `filtered_df` is in scope (e.g. defined in another notebook cell or `%run` import), its index is used. Otherwise the first `MAX_PLOTS` rows of `df` are plotted.

In [8]:
# Filter rows whose hourly NOx emission falls near a target value (kg/h, +/- tolerance).
# `filtered_df` is picked up automatically by the next cell.
TARGET_KG_H       = 400
TOLERANCE_PERCENT = 10

tol = TARGET_KG_H * TOLERANCE_PERCENT / 100
emission_kg = df['NOx Mass (lbs)'] * LBS_TO_KG

mask = (emission_kg >= TARGET_KG_H - tol) & (emission_kg <= TARGET_KG_H + tol)
filtered_df = df.loc[mask].copy()
filtered_df['emission_kg_h']    = emission_kg[mask]
filtered_df['diff_from_target'] = (filtered_df['emission_kg_h'] - TARGET_KG_H).abs()
filtered_df = filtered_df.sort_values('diff_from_target')

print(f'Target: {TARGET_KG_H} kg/h, range {TARGET_KG_H - tol:.1f} - {TARGET_KG_H + tol:.1f} kg/h')
print(f'Found {len(filtered_df):,} matching rows; top 10 closest:')
print(filtered_df[['location', 'utc_time', 'emission_kg_h', 'diff_from_target']].head(10).to_string())

Target: 400 kg/h, range 360.0 - 440.0 kg/h
Found 20,479 matching rows; top 10 closest:
        location                     utc_time  emission_kg_h  diff_from_target
110041      6095  2023-07-29T19:28:04.925000Z     399.977426          0.022574
445592      1364  2020-08-08T18:29:37.274000Z     399.977426          0.022574
502598      8054  2021-02-19T19:12:46.615000Z     399.977426          0.022574
207228      6183  2024-06-06T19:55:18.570000Z     399.977426          0.022574
429834       856  2020-06-22T18:12:19.353000Z     399.977426          0.022574
285944      2291  2019-02-01T18:19:56.599000Z     399.977426          0.022574
464245      6002  2020-10-04T18:59:49.006000Z     399.977426          0.022574
2079        8023  2022-08-02T18:56:40.561000Z     399.977426          0.022574
389577      2291  2020-01-25T19:46:54.668000Z     400.022785          0.022785
595405      1356  2021-12-12T18:24:17.344000Z     400.022785          0.022785


In [9]:
if 'filtered_df' in globals() and not filtered_df.empty:
    all_indices = filtered_df.index.tolist()
    indices = all_indices[:MAX_PLOTS]
    print(f'Found {len(all_indices)} items in filtered_df')
else:
    indices = df.index.tolist()[:MAX_PLOTS]
    print('No filtered_df; using first rows of df')

print(f'Plotting {len(indices)} indices: {indices[:10]}{"..." if len(indices) > 10 else ""}')

Found 20479 items in filtered_df
Plotting 80 indices: [110041, 445592, 502598, 207228, 429834, 285944, 464245, 2079, 389577, 595405]...


## 4. Helper — resolve plant name from coordinates

In [10]:
def resolve_plant_name(plants, ref_lat, ref_lon, idx):
    plant_match = plants[(plants['Latitude'] == ref_lat) & (plants['Longitude'] == ref_lon)]
    if not plant_match.empty:
        return plant_match.iloc[0]['Facility Name']

    geod = Geod(ellps='WGS84')
    min_dist = float('inf')
    closest = None
    for _, plant in plants.iterrows():
        dist = geod.inv(ref_lon, ref_lat, plant['Longitude'], plant['Latitude'])[2]
        if dist < min_dist and dist < 1000:
            min_dist = dist
            closest = plant
    return closest['Facility Name'] if closest is not None else f'Unknown_Plant_{idx}'

## 5. Generate the figures

In [ ]:
print(f"\n{'='*80}\nPlotting {len(indices)} locations\n{'='*80}\n")

for plot_num, idx in enumerate(indices, 1):
    print(f'Processing {plot_num}/{len(indices)}: Index {idx}')
    row = df.iloc[idx]

    swath_path = resolve_swath_path(row['file_path'])
    try:
        with nc.Dataset(swath_path) as ds:
            grp = ds.groups['PRODUCT'] if 'PRODUCT' in ds.groups else ds
            lat = grp['latitude'][:] if grp['latitude'].ndim == 2 else grp['latitude'][0]
            lon = grp['longitude'][:] if grp['longitude'].ndim == 2 else grp['longitude'][0]
            no2_var = grp['nitrogendioxide_tropospheric_column']
            no2 = no2_var[:] if no2_var.ndim == 2 else no2_var[0]
    except Exception as e:
        print(f'   Error loading file: {e}')
        print(f'   Resolved path: {swath_path}')
        continue

    x_min, x_max = row['longitude'] - BOX_DEG, row['longitude'] + BOX_DEG
    y_min, y_max = row['latitude']  - BOX_DEG, row['latitude']  + BOX_DEG

    mask = (lat >= y_min - 0.2) & (lat <= y_max + 0.2) & (lon >= x_min - 0.2) & (lon <= x_max + 0.2)
    if not mask.any():
        print(f'   No data in window for index {idx}')
        continue

    r, c = np.where(mask)
    r0, r1 = max(r.min() - PAD, 0), min(r.max() + PAD, lat.shape[0] - 1)
    c0, c1 = max(c.min() - PAD, 0), min(c.max() + PAD, lon.shape[1] - 1)

    ref_lat = row.get('latitude',  row.get('Latitude'))
    ref_lon = row.get('longitude', row.get('Longitude'))
    ref_name = resolve_plant_name(plants, ref_lat, ref_lon, idx)

    hourly_emission_kg = None
    if 'NOx Mass (lbs)' in row:
        hourly_emission_kg = row['NOx Mass (lbs)'] * LBS_TO_KG
    elif 'hourly_emission' in row:
        hourly_emission_kg = row['hourly_emission'] * LBS_TO_KG

    albedo       = row.get('surface_albedo_nitrogendioxide_window', np.nan)
    solar_zenith = row.get('solar_zenith_angle', row.get('sensor_zenith_angle', np.nan))
    wind_speed   = row.get('wind_speed', np.nan)

    obs_time_utc = pd.to_datetime(row.get('utc_time'), utc=True, errors='coerce')

    fig = plt.figure(figsize=(18, 8))
    gs = gridspec.GridSpec(1, 2, width_ratios=[1, 1], wspace=0.15)

    # Left: satellite basemap
    ax_sat = fig.add_subplot(gs[0, 0])
    for label in ax_sat.get_xticklabels() + ax_sat.get_yticklabels():
        label.set_fontproperties(prop); label.set_fontsize(14)

    dlat = ZOOM_KM / 111.0
    dlon = ZOOM_KM / (111.0 * max(np.cos(np.radians(ref_lat)), 1e-9))
    ax_sat.set_xlim(ref_lon - dlon, ref_lon + dlon)
    ax_sat.set_ylim(ref_lat - dlat, ref_lat + dlat)

    try:
        ctx.add_basemap(ax_sat, crs='EPSG:4326',
                        source=ctx.providers.Esri.WorldImagery,
                        zoom='auto', attribution=False)
        ax_sat.scatter(ref_lon, ref_lat, marker='*', s=400, color='#FF3333',
                       edgecolor='white', linewidth=2.5, zorder=10)
        ax_sat.set_xlabel('Longitude', fontsize=20, fontweight='bold', fontproperties=prop)
        ax_sat.set_ylabel('Latitude', fontsize=20, fontweight='bold', fontproperties=prop)
        ax_sat.set_title('Satellite View', fontweight='bold', pad=10, fontproperties=prop, fontsize=24)
    except Exception as e:
        ax_sat.text(0.5, 0.5, f'Satellite Image\nUnavailable\n({str(e)[:30]}...)',
                    ha='center', va='center', transform=ax_sat.transAxes,
                    fontsize=12, color='#CC0000')
        ax_sat.set_title('Satellite View', fontsize=24, fontweight='bold')

    # Right: TROPOMI NO2
    ax_no2 = fig.add_subplot(gs[0, 1])
    for label in ax_no2.get_xticklabels() + ax_no2.get_yticklabels():
        label.set_fontproperties(prop); label.set_fontsize(14)

    im = ax_no2.pcolormesh(lon[r0:r1+1, c0:c1+1],
                           lat[r0:r1+1, c0:c1+1],
                           no2[r0:r1+1, c0:c1+1],
                           cmap='viridis', vmin=1e-5, vmax=5e-5, zorder=1)
    ax_no2.plot(ref_lon, ref_lat, 'r*', markersize=20,
                markeredgecolor='k', markeredgewidth=1.5, zorder=5)
    ax_no2.set_xlim(x_min, x_max)
    ax_no2.set_ylim(y_min, y_max)

    cbar = plt.colorbar(im, ax=ax_no2, orientation='vertical', pad=0.02, shrink=0.8)
    cbar.set_label(r'NO$_2$ Column (mol/m$^2$)', fontsize=18,
                   fontweight='bold', fontproperties=prop)
    formatter = ScalarFormatter(useMathText=False)
    formatter.set_scientific(True); formatter.set_powerlimits((-2, 2))
    cbar.ax.yaxis.set_major_formatter(formatter)
    cbar.ax.tick_params(labelsize=14)
    for label in cbar.ax.get_yticklabels():
        label.set_fontproperties(prop); label.set_fontsize(14)
    cbar.ax.yaxis.get_offset_text().set_fontproperties(prop)

    ax_no2.set_xlabel('Longitude', fontsize=20, fontweight='bold', fontproperties=prop)
    ax_no2.set_ylabel('Latitude',  fontsize=20, fontweight='bold', fontproperties=prop)
    ax_no2.set_title(r'TROPOMI NO$_2$', fontweight='bold', pad=10,
                     fontproperties=prop, fontsize=24)

    var_text = ''
    if obs_time_utc is not None and not pd.isna(obs_time_utc):
        var_text += f"Observation: {obs_time_utc.strftime('%Y-%m-%d %H:%M UTC')}\n"
    if hourly_emission_kg is not None and not np.isnan(hourly_emission_kg):
        var_text += f'Hourly Emission: {hourly_emission_kg:.2f} kg/h\n'
    else:
        var_text += 'Hourly Emission: N/A\n'
    if not np.isnan(wind_speed):
        var_text += f'Wind Speed: {wind_speed:.2f} m/s\n'
    else:
        var_text += 'Wind Speed: N/A\n'
    if not np.isnan(albedo):
        var_text += r'Surface Albedo (NO$_2$ Window)' + f': {albedo:.3f}\n'
    else:
        var_text += r'Surface Albedo ($NO_2$ Window): N/A' + '\n'
    if not np.isnan(solar_zenith):
        var_text += f'Solar Zenith Angle: {solar_zenith:.1f}\u00b0'
    else:
        var_text += 'Solar Zenith Angle: N/A'

    ax_no2.text(0.98, 0.98, var_text,
                transform=ax_no2.transAxes,
                fontsize=16, verticalalignment='top', horizontalalignment='right',
                bbox=dict(boxstyle='round,pad=0.6', facecolor='white',
                          alpha=0.92, edgecolor='black', linewidth=1.5),
                fontproperties=prop, zorder=10)

    fig.suptitle(f'{ref_name}\nLocation: ({ref_lat:.2f}, {ref_lon:.2f})',
                 fontsize=24, fontweight='bold', y=0.98, fontproperties=prop)

    output_filename = os.path.join(OUTPUT_DIR, f'no2_enhanced_{idx}.pdf')
    plt.savefig(output_filename, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()

    if hourly_emission_kg is not None and not np.isnan(hourly_emission_kg):
        print(f'   Hourly Emission: {hourly_emission_kg:.2f} kg/h')
    print(f'   Wind Speed: {wind_speed:.2f} m/s' if not np.isnan(wind_speed) else '   Wind Speed: N/A')
    print(f'   Albedo: {albedo:.3f}'              if not np.isnan(albedo)       else '   Albedo: N/A')
    print(f'   Solar Zenith: {solar_zenith:.1f}\u00b0' if not np.isnan(solar_zenith) else '   Solar Zenith: N/A')
    print(f'   Saved: {output_filename}\n')

print(f"\n{'='*80}\nDone. Wrote figures to {OUTPUT_DIR}\n{'='*80}\n")
